# Load and convert Impect data to SPADL

Use getter=local with Impect open-data, or getter=remote with IMPECT_USER_1 / IMPECT_PASS_1 in .env.

In [ ]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from socceraction.data.impect import ImpectLoader
import socceraction.spadl as spadl
load_dotenv()

In [ ]:
IML = ImpectLoader(getter="local", root="../tests/datasets/impect/raw")
competitions = IML.competitions()
competition_id = int(competitions.iloc[0]["competition_id"])
season_id = int(competitions.iloc[0]["season_id"])
games = IML.games(competition_id=competition_id, season_id=season_id)
game_id = int(games.iloc[0]["game_id"])
home_team_id = int(games.loc[games.game_id == game_id, "home_team_id"].iloc[0])
events = IML.events(game_id)
actions = spadl.impect.convert_to_actions(events, home_team_id)
actions = spadl.add_names(actions)
actions.type_name.value_counts().head(15)

## Batch SPADL for a full iteration

Build `data/impect/<output_basename>/spadl-impect.h5` with the **local pipeline** (see `docs/documentation/data/impect_pipeline.rst`). Scripts live in `private/impect-pipeline/` (gitignored).

Use open-data locally (below) or cache from the API with `--getter remote --cache`.


In [ ]:
# Local open-data (11 matches with events in tests/datasets/impect/raw)
!python private/impect-pipeline/build_spadl_h5.py \
  --config docs/documentation/data/impect_open_data.example.json \
  --getter local \
  --data-root tests/datasets/impect/raw


In [ ]:
import pandas as pd
spadl_h5 = Path('data/impect/impect-open-data/spadl-impect.h5')
with pd.HDFStore(spadl_h5) as store:
    n_actions = sum(1 for k in store.keys() if k.startswith('/actions/'))
    print('games in schedule:', len(store['games']))
    print('games with SPADL:', n_actions)
    sample = store['actions/game_122838']
print(sample.type_name.value_counts().head(8))


### Remote API (one iteration)

Requires `.env` with `IMPECT_USER_1` / `IMPECT_PASS_1`. Use `--cache` to store JSON under `data/impect/{iteration_id}/` before conversion.

In [ ]:
# Uncomment after copying docs/.../impect_iteration.example.json to private/impect-pipeline/config/my_iteration.json
# !python private/impect-pipeline/build_spadl_h5.py \
#   --config private/impect-pipeline/config/my_iteration.json \
#   --getter remote --cache
